# Summarizer 
Used to aggregate results from BCRT experiments, which are stored in JSON files. Specifically this is needed for the BCRT ECDF case, but can also be used for KDE simulation. 

In [ ]:
import json
import numpy as np
import pandas as pd
from IPython.display import display

# ============================================================
# Robust JSON parsing + key inference
# ============================================================

def _extract_all_results(payload):
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        for k in ("all_results", "results", "runs", "data"):
            if k in payload and isinstance(payload[k], list):
                return payload[k]
    raise ValueError("Unrecognized JSON structure.")

def _norm_key(k: str) -> str:
    return "".join(ch for ch in k.lower() if ch.isalnum())

def _find_key(d: dict, candidates):
    if not d:
        return None

    nk_map = {_norm_key(k): k for k in d.keys()}

    flat = []
    for c in candidates:
        if isinstance(c, (list, tuple)):
            flat.extend(list(c))
        else:
            flat.append(c)

    for cand in flat:
        nk = _norm_key(cand)
        if nk in nk_map:
            return nk_map[nk]
    return None

def summarize_rates_from_payload(payload, require_slopes=False):
    all_results = _extract_all_results(payload)
    if not all_results:
        raise ValueError("No results found in JSON.")

    sample = all_results[0]

    key_alpha = _find_key(sample, ["alpha_real", "alpha"])
    key_q     = _find_key(sample, ["bias_rate", "q"])
    key_p     = _find_key(sample, ["p_target", "p"])

    key_slopes_w1   = _find_key(sample, [("slopes_W1", "slopes_w1", "W1_slopes", "w1_slopes")])
    key_slopes_mmd2 = _find_key(sample, [("slopes_MMD2", "slopes_mmd2", "MMD2_slopes", "mmd2_slopes")])

    key_mean_w1   = _find_key(sample, [("mean_slope_W1", "mean_slope_w1")])
    key_mean_mmd2 = _find_key(sample, [("mean_slope_MMD2", "mean_slope_mmd2")])

    missing_core = [name for name, kk in [
        ("alpha_real", key_alpha),
        ("bias_rate", key_q),
        ("p_target", key_p)
    ] if kk is None]

    if missing_core:
        raise KeyError(f"Missing required fields: {missing_core}. "
                       f"Available keys: {sorted(sample.keys())}")

    keys = sorted({(r[key_alpha], r[key_q], r[key_p]) for r in all_results})

    rows = []
    for a, q, p in keys:
        grp = [r for r in all_results
               if r[key_alpha] == a and r[key_q] == q and r[key_p] == p]

        # ---- W1 ----
        if key_slopes_w1 is not None:
            slopes_w1 = np.concatenate(
                [np.asarray(r[key_slopes_w1], dtype=float) for r in grp]
            )
            w1_mean = float(-np.mean(slopes_w1))
            w1_med  = float(-np.median(slopes_w1))
        elif key_mean_w1 is not None:
            w1_mean = float(-np.mean([float(r[key_mean_w1]) for r in grp]))
            w1_med  = np.nan
        else:
            w1_mean = np.nan
            w1_med  = np.nan

        # ---- MMD (from MMD^2) ----
        if key_slopes_mmd2 is not None:
            slopes_mmd2 = np.concatenate(
                [np.asarray(r[key_slopes_mmd2], dtype=float) for r in grp]
            )
            slopes_mmd = 0.5 * slopes_mmd2
            mmd_mean = float(-np.mean(slopes_mmd))
            mmd_med  = float(-np.median(slopes_mmd))
        elif key_mean_mmd2 is not None:
            mean_slope_mmd2 = float(np.mean([float(r[key_mean_mmd2]) for r in grp]))
            mmd_mean = float(-(0.5 * mean_slope_mmd2))
            mmd_med  = np.nan
        else:
            mmd_mean = np.nan
            mmd_med  = np.nan

        rows.append({
            "alpha": float(a),
            "q": float(q),
            "p": float(p),
            "theory": float(min(float(p), float(a), float(q))),
            "W1_mean": w1_mean,
            "W1_med": w1_med,
            "MMD_mean": mmd_mean,
            "MMD_med": mmd_med,
        })

    df = pd.DataFrame(rows).sort_values(["alpha", "q", "p"]).reset_index(drop=True)
    return df


# ============================================================
# Path to results JSON 
# ============================================================

JSON_PATH = "logs/KDE/BCRT_quick_20260216-123456/all_results.json"

with open(JSON_PATH, "r", encoding="utf-8") as f:
    payload = json.load(f)

df = summarize_rates_from_payload(payload, require_slopes=False)
display(df)
